# LangSmith

In [1]:
import os
import warnings
from pathlib import Path

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [4]:
go_up_from_current_directory(go_up=2)

from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


### Default Tracing

- LangSmith traces a lot of data without us needing to do anything.

In [8]:
from langchain_openai import ChatOpenAI

model_str: str = "google/gemini-2.0-flash-001"
# Deterministic responses
llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str,
)

console.print(llm.invoke("Tell me a short joke about Lionel Messi."))

AIMessage(
    content="Why did Messi get a bad grade in his history class?\n\nBecause he kept talking about the past... and 
how many Ballon d'Ors he had!\n",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 32,
            'prompt_tokens': 9,
            'total_tokens': 41,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'google/gemini-2.0-flash-001',
        'system_fingerprint': None,
        'id': 'gen-1753812377-mkAXtlXHeNFCyoGSnsCP',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--5a06ab74-1cc3-48a3-9888-205e7e9e2485-0',
    usage_metadata={
        'input_tokens': 9,
        'output_tokens': 32,
        'total_tokens': 41,
        'input_token_details': {},
        'output_token_details': {}
    }
)

<br>

### LangSmith UI

<img src="../../static-files/01-langSmith.png" alt="LangSmith UI" width="600">

## Tracing Non-LangChain Code

In [9]:
import random
import time

from langsmith import traceable


@traceable
def generate_random_number() -> int:
    """Generate a random number between 0 and 100."""
    return random.randint(0, 100)


@traceable
def generate_string_delay(input_str: str) -> str:
    """Generate a string with a random delay."""
    number = random.randint(1, 5)
    time.sleep(number)
    return f"{input_str} ({number})"


@traceable
def random_error() -> str:
    """Generate a random error or return a success message."""
    number = random.randint(0, 1)
    if number == 0:
        raise ValueError("Random error")
    return "No error"

In [11]:
from tqdm import tqdm

with tqdm(total=3, desc="Processing", unit="step") as pbar:
    generate_random_number()
    pbar.update(1)
    generate_string_delay("Hello")
    pbar.update(1)
    try:
        random_error()
    except ValueError:
        pass
    pbar.update(1)

Processing: 100%|██████████| 3/3 [00:04<00:00,  1.34s/step]
